In [ ]:
# Example 1: InMemoryBroker + simple task registration
from taskiq import InMemoryBroker

broker = InMemoryBroker()

@broker.task
async def add(a: int, b: int) -> int:
    return a + b

print(f"broker = {broker}")
print(f"add task name = {add.task_name}")
print(f"registered tasks = {list(broker.get_all_tasks().keys())}")
assert add.task_name in broker.get_all_tasks()

In [ ]:
# Example 2: kick a task + wait for result via InMemoryBroker
from taskiq import InMemoryBroker

broker = InMemoryBroker()

@broker.task
async def multiply(x: int, y: int) -> int:
    return x * y

await broker.startup()
task = await multiply.kiq(6, 7)
result = await task.wait_result(timeout=5)
await broker.shutdown()

print(f"task_id = {task.task_id}")
print(f"return_value = {result.return_value}")
print(f"is_err = {result.is_err}")
assert result.return_value == 42
assert not result.is_err

In [ ]:
# Example 3: TaskiqScheduler + label-based scheduling
# taskiq 0.12.x: LabelScheduleSource collects the schedule labels during startup(),
# so the source must be started before get_schedules() returns anything.
from taskiq import InMemoryBroker, TaskiqScheduler
from taskiq.schedule_sources import LabelScheduleSource

broker = InMemoryBroker()

@broker.task(schedule=[{"cron": "*/5 * * * *"}])
async def periodic() -> str:
    return "tick"

scheduler = TaskiqScheduler(broker=broker, sources=[LabelScheduleSource(broker)])
print(f"scheduler = {scheduler}")
print(f"source count = {len(scheduler.sources)}")
await scheduler.sources[0].startup()
schedules = await scheduler.sources[0].get_schedules()
print(f"schedules discovered = {len(schedules)}")
assert len(schedules) >= 1
assert schedules[0].cron == "*/5 * * * *"
print(f"first schedule: task={schedules[0].task_name} cron={schedules[0].cron}")

In [ ]:
# Example 4: TaskiqMessage + serializer round-trip
from taskiq import TaskiqMessage
from taskiq.serializers import JSONSerializer

msg = TaskiqMessage(
    task_id="abc-123",
    task_name="demo",
    labels={},
    labels_types=None,
    args=[1, 2],
    kwargs={"k": "v"},
)
ser = JSONSerializer()
payload = ser.dumpb(msg.model_dump())
decoded = ser.loadb(payload)
print(f"payload bytes = {len(payload)}")
print(f"decoded task_id = {decoded['task_id']}")
print(f"decoded args = {decoded['args']}")
assert decoded["task_id"] == "abc-123"
assert decoded["args"] == [1, 2]

In [ ]:
# Example 5: middleware + dependency injection via TaskiqDepends
from taskiq import InMemoryBroker, TaskiqDepends, TaskiqMiddleware

calls = {"pre": 0, "post": 0}

class CountingMiddleware(TaskiqMiddleware):
    async def pre_execute(self, message):
        calls["pre"] += 1
        return message
    async def post_execute(self, message, result):
        calls["post"] += 1

def get_greeting() -> str:
    return "hello"

broker = InMemoryBroker().with_middlewares(CountingMiddleware())

@broker.task
async def greet(name: str, greeting: str = TaskiqDepends(get_greeting)) -> str:
    return f"{greeting}, {name}"

await broker.startup()
result = await (await greet.kiq("world")).wait_result(timeout=5)
await broker.shutdown()
print(f"return = {result.return_value}")
print(f"middleware calls = {calls}")
assert result.return_value == "hello, world"
assert calls["pre"] >= 1 and calls["post"] >= 1